# <p style="text-align: center;"> Legal Entity Identifier (LEI) Notebooks </p>
### <p style="text-align: center;">Legal Entity Events</p>


#### Tasks covered in this notebook:
- Download latest Golden Copy file
- Retrieve previous values before a Legal Entity Event occurred by using the GLEIF API
- Retrieve Legal Entity Events of retired entities using the Golden Copy
- Time difference between event occurrence and reporting

Details regarding Legal Entity Events can be found here: https://dq-dashboard.gleif.org/legal-entity-events


In [1]:
# This cell determines whether the notebook is run in Google Colab.

try:
    import google.colab
    IN_COLAB = True
    
    # Connect your Google Drive to the Colab environment:
    from google.colab import drive
    drive.mount('/content/drive')

    # Adjust the path:
    import sys
    sys.path.append('/content/drive/MyDrive/gleif_notebooks') # Adjust this path, if necessary

except ImportError:
    IN_COLAB = False


In [2]:
# =============================================================================
# CONFIGURATION OPTIONS
# =============================================================================

# DATA SCOPE OPTIONS
USE_FULL_DATASET = False  # Set to 'False' to use only essential data elements as defined below. 'True' will use all data elements.

# STORAGE OPTIONS
SAVE_TO_DISK = False  # Set to 'False' to keep data only in memory. 'True' will store LEI data and mappings locally

# DISPLAY CONFIGURATION
print("=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"Data Scope: {'Full Dataset' if USE_FULL_DATASET else 'Essential Columns Only'}")
print(f"Storage: {'Save to Disk' if SAVE_TO_DISK else 'Keep in Memory Only'}")

print("=" * 60)


CONFIGURATION SUMMARY
Data Scope: Essential Columns Only
Storage: Keep in Memory Only


In [3]:
# Import required libraries and utils
from datetime import datetime
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import time

# Import from utils (this will automatically run environment setup)
from utils import (
    GoldenCopyDownload,
    GLEIFAPI,
    LegalEntityEventsVisualizer,
    ColumnNames
)

# Only use data element that are relevant for Legal Entity Event analysis
REQUIRED_COLUMNS = ColumnNames.ESSENTIAL_COLUMNS + ColumnNames.EVENT_COLUMNS + ColumnNames.SUCCESSOR_COLUMNS + ColumnNames.REGISTRATION_COLUMNS

if not USE_FULL_DATASET:
    print(f"Selected Columns: {len(REQUIRED_COLUMNS)} columns")

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

Selected Columns: 37 columns
Environment: Local


As a first step, we pull the level 1 Golden Copy LEI data. Depending on the configuration and your system, this may take a while.

In [ ]:
date = datetime.today().strftime("%Y-%m-%d")

# Alternatively, use specific date and time of day:

# date = '2025-12-03' 
# time_preference = "00:00" # Golden Copy is available daily at 00:00, 08:00, 16:00 UTC

# Base data directory (relative to current working directory in notebook)
DEFAULT_DATA_DIR = Path.cwd() 

# Subfolder for GC downloads
DEFAULT_GC_DIR = DEFAULT_DATA_DIR / "gc_downloads"

# Subfolder for mappings 
DEFAULT_SAVE_DIR = DEFAULT_DATA_DIR / "downloads"

# Download data using the selected setup
level_1_data = await GoldenCopyDownload.download_with_config(
    date=date,
    save_to_disk=SAVE_TO_DISK,
    use_full_dataset=USE_FULL_DATASET,
    essential_columns=REQUIRED_COLUMNS if not USE_FULL_DATASET else None,
    save_dir=DEFAULT_GC_DIR,
    #time=time_preference,
)

Using subset of 37 columns
Found URL for: https://goldencopy.gleif.org/api/v2/golden-copies/publishes/lei2/20251204-0000.csv


In [ ]:
# Sneak peek at LEI data:
level_1_data.head(3)

,LEI,Entity.LegalName,Registration.RegistrationStatus,Entity.LegalEntityEvents.LegalEntityEvent.1.LegalEntityEventType,Entity.LegalEntityEvents.LegalEntityEvent.1.event_status,Entity.LegalEntityEvents.LegalEntityEvent.1.LegalEntityEventEffectiveDate,Entity.LegalEntityEvents.LegalEntityEvent.1.LegalEntityEventRecordedDate,Entity.LegalEntityEvents.LegalEntityEvent.2.LegalEntityEventType,Entity.LegalEntityEvents.LegalEntityEvent.2.event_status,Entity.LegalEntityEvents.LegalEntityEvent.2.LegalEntityEventEffectiveDate,...,Entity.SuccessorEntity.3.SuccessorEntityName,Entity.SuccessorEntity.3.SuccessorLEI,Entity.SuccessorEntity.4.SuccessorEntityName,Entity.SuccessorEntity.4.SuccessorLEI,Entity.SuccessorEntity.5.SuccessorEntityName,Entity.SuccessorEntity.5.SuccessorLEI,Registration.InitialRegistrationDate,Registration.ManagingLOU,Registration.NextRenewalDate,Registration.LastUpdateDate
0,001GPB6A9XPE8XJICC14,Fidelity Advisor Leveraged Company Stock Fund,LAPSED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2012-11-29T16:33:00.000Z,5493001KJTIIGC8Y1R12,2025-04-18T15:48:53.604Z,2025-04-19T13:30:05.472Z
1,004L5FPTUREIWK9T2N63,"Hutchin Hill Capital, LP",LAPSED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2012-06-06T15:56:00.000Z,5493001KJTIIGC8Y1R12,2018-05-08T13:46:00.000Z,2023-08-04T15:31:41.430Z
2,00EHHQ2ZHDCFXJCPCL46,Vanguard Fiduciary Trust Company Vanguard Russ...,ISSUED,CHANGE_LEGAL_NAME,COMPLETED,2025-05-14T00:00:00+02:00,2025-05-14T08:51:02+02:00,CHANGE_LEGAL_NAME,COMPLETED,2025-05-08T00:00:00+02:00,...,NaN,NaN,NaN,NaN,NaN,NaN,2012-10-05T22:30:00+02:00,5299000J2N45DDNE4Y28,2026-06-24T11:39:46+02:00,2025-05-14T11:17:36+02:00


### Retrieve previous values before Legal Entity Event occurred by using the GLEIF API

This section shows how to access Legal Entity Events through the GLEIF API for specific LEIs and Legal Entity Event Types to view the previous and new values for data fields that have been updated due to a Legal Entity Event.

First, let's initialize the GLEIF API client.

In [ ]:
# Initialize API client
api_client = GLEIFAPI()

GLEIF API client initialized


Below, we implemented a class containing useful methods for analyzing Legal Entity Events by using the GLEIF API. 

The implementation includes special treatment of certain event types, so that only relevant reference data updates are returned.

In [ ]:
class LegalEntityEventsAnalyzerAPI:
    """
    Analyzer for Legal Entity Events (LEEs) in LEI data using the GLEIF API.
    Fetches field modifications from the API and provides helpers to get event tables and delay metadata.
    """

    DEFAULT_REQUIRED_FIELD = "lei:LegalEntityEventType"
    DEFAULT_PAGE_LIMIT = 200
    DEFAULT_MAX_PAGES = 10  # fetches given number of pages

    # Default behaviour (for dissolution, liquidation or similar events)
    DEFAULT_STATUS_FIELDS = [
        "lei:RegistrationStatus",
        "lei:EntityStatus",
        "lei:LegalEntityEventEffectiveDate",
        "lei:LegalEntityEventRecordedDate",
        "lei:SuccessorLEI",
        "lei:SuccessorEntityName",
        "lei:EntityLegalName",
        "@field_xpath",
        "@event_status",
    ]

    # Date fields used for delay calculation
    DATE_FIELDS = [
        "lei:LegalEntityEventEffectiveDate",
        "lei:LegalEntityEventRecordedDate",
    ]
    
    SPECIAL_EVENT_RULES = {
        "CHANGE_LEGAL_NAME": {
            "lei:LegalName",
            "lei:LegalName.@xml:lang",
        },
        "CHANGE_OTHER_NAMES": {
            "lei:OtherEntityName",
            "@xml:lang",

        },
        "CHANGE_LEGAL_ADDRESS": {
            # API returns simplified field names without LegalAddress prefix
            "lei:FirstAddressLine",
            "lei:AdditionalAddressLine",
            "lei:AdditionalAddressLine.1",
            "lei:AdditionalAddressLine.2",
            "lei:AdditionalAddressLine.3",
            "lei:City",
            "lei:Region",
            "lei:Country",
            "lei:PostalCode",
        },
        "CHANGE_HQ_ADDRESS": {
            # API returns simplified field names without HeadquartersAddress prefix
            "lei:FirstAddressLine",
            "lei:AdditionalAddressLine",
            "lei:AdditionalAddressLine.1",
            "lei:AdditionalAddressLine.2",
            "lei:AdditionalAddressLine.3",
            "lei:City",
            "lei:Region",
            "lei:Country",
            "lei:PostalCode",
        },
        "CHANGE_LEGAL_FORM": {
            "lei:LegalForm",
            "lei:EntityLegalFormCode",
            "lei:OtherLegalForm",
        },
        "TRANSFORMATION_BRANCH_TO_SUBSIDIARY": {
            "rr:RegistrationStatus",
            "rr:RelationshipType",
            "rr:EndNode",
            "repex:ExceptionReason",
            "repex:ExceptionCategory"
        },
        "TRANSFORMATION_SUBSIDIARY_TO_BRANCH": {
            "rr:RegistrationStatus",
            "rr:RelationshipType",
            "rr:EndNode",
            "rr:NodeID",
            "repex:ExceptionReason",
            "repex:ExceptionCategory"
        },
        "ACQUISITION_BRANCH": {
            "rr:RegistrationStatus",
            "rr:RelationshipType",
            "rr:EndNode",
            "rr:NodeID",
            "repex:ExceptionReason",
            "repex:ExceptionCategory"
        },
        "TRANSFORMATION_UMBRELLA_TO_STANDALONE": {
            "rr:RegistrationStatus",
            "rr:RelationshipType",
            "rr:EndNode",
            "rr:NodeID",
            "repex:ExceptionReason",
            "repex:ExceptionCategory"
        },
        "REVERSE_TAKEOVER": {
            "rr:RegistrationStatus",
            "rr:RelationshipType",
            "rr:EndNode",
            "rr:NodeID",
            "repex:ExceptionReason",
            "repex:ExceptionCategory"
        }   
    }


    def __init__(
        self,
        api_client,
        leis: List[str],
        page_limit: int = DEFAULT_PAGE_LIMIT,
        max_pages: int = DEFAULT_MAX_PAGES,
        required_field: str = DEFAULT_REQUIRED_FIELD,
    ) -> None:
        self.api_client = api_client
        self.leis = leis
        self.page_limit = page_limit
        self.max_pages = max_pages
        self.required_field = required_field

        # state
        self.lei_data_dict: Dict[str, pd.DataFrame] = {}

    # ---------- Internal helpers (data only) ----------

    def _get_event_types(self, day_df: pd.DataFrame) -> Dict[str, str]:
        """
        Return a mapping for all event rows on the selected day:
        {EVENT_TYPE_UPPER: original_label}
        """
        rows = day_df[day_df["Field"] == self.required_field].copy()
        rows["event_upper"] = rows["New value"].astype(str).str.upper()
        mapping = (
            rows.drop_duplicates("event_upper")
            .set_index("event_upper")["New value"]
            .to_dict()
        )
        return mapping

    def _build_field_mask_for_event(
        self, day_df: pd.DataFrame, event_upper: str
        ) -> pd.Series:
        """
        For a single event type on this day, return a boolean mask of rows to keep:
        - always keep the event row itself
        - plus fields selected by SPECIAL_EVENT_RULES or default status fields
        - always include DATE_FIELDS to allow delay calculation for all event types
        """
        fields = day_df["Field"]

        # event row: LegalEntityEventType with this specific new_value
        event_row_mask = (fields == self.required_field) & (
            day_df["New value"].astype(str).str.upper() == event_upper
        )

        mask = event_row_mask.copy()

        # Choose rule based on prefix match
        rule = None
        for prefix, field_set in self.SPECIAL_EVENT_RULES.items():
            if event_upper.startswith(prefix):
                rule = field_set
                break

        if rule is not None:
            mask |= fields.isin(rule)
        else:
            # Default behaviour
            mask |= fields.isin(self.DEFAULT_STATUS_FIELDS)

        # Always keep date fields so delay is available for all event types
        mask |= fields.isin(self.DATE_FIELDS)

        return mask

    @staticmethod
    def _is_empty_value(val) -> bool:
        """Treat these as empty/deleted."""
        if val is None:
            return True
        s = str(val).strip()
        return s == "" or s.upper() in {"N/A", "NONE", "NULL", "NAN"}

    @staticmethod
    def _pick_final_new_value_ignore_removed(day_df: pd.DataFrame, field_name: str):
        """
        For a given day_df (single updated_date) and field_name, return the final
        New value for that field on that day, ignoring values that were removed
        later on the same day.
        """

        rows = day_df[day_df["Field"] == field_name].copy()
        if rows.empty:
            return None

        # Stable order within the day
        rows["_row_order"] = np.arange(len(rows))
        if "modification_date" in rows.columns:
            rows = rows.sort_values(["modification_date", "_row_order"])
        else:
            rows = rows.sort_values("_row_order")

        removed_vals = set()

        # check the removed values
        for _, r in rows.iterrows():
            prev_raw = r["Previous value"]
            new_raw = r["New value"]

            prev_empty = LegalEntityEventsAnalyzerAPI._is_empty_value(prev_raw)
            new_empty  = LegalEntityEventsAnalyzerAPI._is_empty_value(new_raw)

            prev_norm = None if prev_empty else str(prev_raw).strip()
            new_norm  = None if new_empty  else str(new_raw).strip()

            # deletion row: New is empty, Previous is some real value
            if new_norm is None and prev_norm is not None:
                removed_vals.add(prev_norm)

        # new value that has NOT been removed during the day.
        chosen = None
        for _, r in rows.iloc[::-1].iterrows():
            new_raw = r["New value"]
            if LegalEntityEventsAnalyzerAPI._is_empty_value(new_raw):
                continue

            new_norm = str(new_raw).strip()
            if new_norm in removed_vals:
                # this value was removed at some point → skip
                continue

            chosen = new_raw
            break

        return chosen

    # ---------- Data fetch ----------

    def _fetch_data(self) -> None:
        """Populate self.lei_data_dict with field modifications per LEI."""
        self.lei_data_dict = {}

        for lei in self.leis:
            print(f"Fetching field modifications for LEI: {lei}")

            df_lei = self.api_client.fetch_field_modifications(
                lei=lei,
                page_limit=self.page_limit,
                max_pages=self.max_pages,
            )

            if df_lei.empty:
                self.lei_data_dict[lei] = pd.DataFrame()
                continue

            df_lei["modification_date"] = pd.to_datetime(
                df_lei["modification_date"], errors="coerce"
            )
            df_lei = df_lei.dropna(subset=["modification_date"])
            df_lei["updated_date"] = df_lei["modification_date"].dt.strftime("%Y-%m-%d")

            self.lei_data_dict[lei] = df_lei

        print(f"\nData fetched for {len(self.lei_data_dict)} LEI(s)")

    def fetch_data(self) -> None:
        """Public wrapper for data fetch."""
        self._fetch_data()

    # ---------- Public data helpers ----------

    def get_lei_dataframe(self, lei: str) -> pd.DataFrame:
        """Return the stored DataFrame for a given LEI (empty if missing)."""
        return self.lei_data_dict.get(lei, pd.DataFrame())

    def get_valid_dates_for_lei(self, lei: str) -> List[str]:
        """All dates where the required_field was modified for this LEI."""
        df_lei = self.get_lei_dataframe(lei)
        if df_lei.empty:
            return []

        valid_dates = (
            df_lei[df_lei["Field"] == self.required_field]["updated_date"]
            .dropna()
            .unique()
            .tolist()
        )
        return sorted(valid_dates)

    
    def get_event_tables_for_day(
        self,
        lei: str,
        selected_date: str,
        event_types: Optional[List[str]] = None,
    ) -> Dict[str, pd.DataFrame]:
        """
        For a given LEI + date, return {event_label: visible_df_for_that_event}
        where visible_df contains [Field, Previous value, New value].

        If `event_types` is provided, only those event types are included.
        e.g. ["CHANGE_LEGAL_NAME", "CHANGE_LEGAL_ADDRESS"]
        """
        df_lei = self.get_lei_dataframe(lei)
        if df_lei.empty:
            return {}

        if selected_date not in df_lei["updated_date"].unique():
            return {}

        day_df = df_lei[df_lei["updated_date"] == selected_date]

        if self.required_field not in day_df["Field"].unique():
            return {}

        event_map = self._get_event_types(day_df)
        result: Dict[str, pd.DataFrame] = {}

        # Normalise requested event types to UPPER for comparison
        allowed_upper = None
        if event_types:
            allowed_upper = {e.upper() for e in event_types}

        for event_upper, event_label in event_map.items():
            # If filter is set, skip events that are not requested
            if allowed_upper is not None and event_upper not in allowed_upper:
                continue

            mask = self._build_field_mask_for_event(day_df, event_upper)
            event_df = day_df[mask].copy()

            event_df["__is_event"] = event_df["Field"] == self.required_field
            event_df = event_df.sort_values(
                ["__is_event", "Field"], ascending=[False, True]
            )
        
            visible_df = event_df[
                ~event_df["Field"].isin(self.DATE_FIELDS + [self.required_field])
            ][["Field", "Previous value", "New value"]].reset_index(drop=True)
        
            result[event_label] = visible_df
        return result


    def get_delay_metadata_for_day(
        self, lei: str, selected_date: str
    ) -> Tuple[str, str, str]:
        """Return (recorded_str, effective_str, delay_str) for the given LEI+date."""
        df_lei = self.get_lei_dataframe(lei)
        if df_lei.empty or selected_date not in df_lei["updated_date"].unique():
            return ("n/a", "n/a", "n/a")
    
        day_df = df_lei[df_lei["updated_date"] == selected_date].copy()
    
        # Ignore values that were removed later that same day
        effective_raw = self._pick_final_new_value_ignore_removed(
            day_df, "lei:LegalEntityEventEffectiveDate"
        )
        recorded_raw = self._pick_final_new_value_ignore_removed(
            day_df, "lei:LegalEntityEventRecordedDate"
        )
        # Parse to datetime
        ed_parsed = (
            pd.to_datetime(effective_raw, errors="coerce")
            if effective_raw
            else pd.NaT
        )
        rd_parsed = (
            pd.to_datetime(recorded_raw, errors="coerce")
            if recorded_raw
            else pd.NaT
        )
        recorded_str = (
            rd_parsed.strftime("%Y-%m-%d")
            if pd.notnull(rd_parsed)
            else (recorded_raw if recorded_raw else "n/a")
        )
        effective_str = (
            ed_parsed.strftime("%Y-%m-%d")
            if pd.notnull(ed_parsed)
            else (effective_raw if effective_raw else "n/a")
        )
        delay_str = "n/a"
        if pd.notnull(rd_parsed) and pd.notnull(ed_parsed):
            try:
                delay_days = (rd_parsed - ed_parsed).days
                delay_str = f"{delay_days} day(s)"
            except Exception:
                delay_str = "n/a"
        return recorded_str, effective_str, delay_str



Using the LegalEntityEventsAnalyzerAPI class, we specify a list of LEIs to be in scope. Optionally, a list of Legal Entity Event Types can be chosen. If no event types are selected, the code will return information about all event types.

In [ ]:
EXAMPLE_LEIS = [
    "ZX7KYZN3LRZ6GZGZ0D51",
    "YL98Y8FYIV7DWE5KIJ84",
    "097900BHH10000079959",
    "097900BIBX0000127073",
    "2549005WL1W9GJ5TM121",
    "7437006AOVHI6NONEV08",
]

print(f"Example LEIs: {EXAMPLE_LEIS}")

# Initialize LEE analyzer
analyzer = LegalEntityEventsAnalyzerAPI(api_client, leis=EXAMPLE_LEIS)

# Fetch data with progress indication
print("Fetching field modifications from GLEIF API...")

start_time = time.time()

analyzer.fetch_data()  # This will show progress messages for each LEI

elapsed = time.time() - start_time
print(f"✓ Data fetched in {elapsed:.1f} seconds\n")

# Choose specific event types you would like to filter
event_types_filter = [
    "CHANGE_LEGAL_NAME",
    # "CHANGE_OTHER_NAMES",
    "CHANGE_LEGAL_ADDRESS",
    "CHANGE_HQ_ADDRESS",
    "CHANGE_LEGAL_FORM",
    # "ACQUISITION_BRANCH",
    "TRANSFORMATION_BRANCH_TO_SUBSIDIARY",
    # "TRANSFORMATION_SUBSIDIARY_TO_BRANCH",
    # "TRANSFORMATION_UMBRELLA_TO_STANDALONE",
    "DISSOLUTION",
    # "BREAKUP",
    "MERGERS_AND_ACQUISITIONS",
    # "DEMERGER",
    # "SPINOFF",
    # "BANKRUPTCY",
    "LIQUIDATION",
    # "VOLUNTARY_ARRANGEMENT",
    # "INSOLVENCY",
    "ABSORPTION",
    # "REVERSE_TAKEOVER"
]

# Initialize LEE visualizer to display the data
viz = LegalEntityEventsVisualizer(
    data_provider=analyzer,
    event_types=event_types_filter,
)

viz.display()


Example LEIs: ['ZX7KYZN3LRZ6GZGZ0D51', 'YL98Y8FYIV7DWE5KIJ84', '097900BHH10000079959', '097900BIBX0000127073', '2549005WL1W9GJ5TM121', '7437006AOVHI6NONEV08']
Fetching field modifications from GLEIF API...
Fetching field modifications for LEI: ZX7KYZN3LRZ6GZGZ0D51
Fetching field modifications for LEI: YL98Y8FYIV7DWE5KIJ84
Fetching field modifications for LEI: 097900BHH10000079959
Fetching field modifications for LEI: 097900BIBX0000127073
Fetching field modifications for LEI: 2549005WL1W9GJ5TM121
Fetching field modifications for LEI: 7437006AOVHI6NONEV08

Data fetched for 6 LEI(s)
✓ Data fetched in 12.8 seconds



HTML(value='Retrieve the previous value that was used before a given LegalEntityEvent was triggered for a sele…

Dropdown(description='LEI:', layout=Layout(width='350px'), options=('ZX7KYZN3LRZ6GZGZ0D51', 'YL98Y8FYIV7DWE5KI…

Dropdown(description='Modification Date:', layout=Layout(width='350px'), options=(), style=DescriptionStyle(de…

Output()

The above widget allows selecting an LEI of interest and the date on which an event was recorded in the LEI repository. The table indicates the field values before and after each event.

### Retrieve Legal Entity Events of retired entities using the Golden Copy

Instead of using the GLEIF API, it is also possible to retrieve the list of Legal Entity Events using the Golden Copy Files. However, the Golden Copy will not show the previous value before an event occurred. 

In this example, we showcase how to retrieve Legal Entity Events and, if applicable, successor entity information of retired entities. In the LEI repository, retired entities are marked with RegistrationStatus "RETIRED".

In [ ]:
retired_level_1 = level_1_data.loc[level_1_data['Registration.RegistrationStatus']=='RETIRED'].copy()

print("The total records with RETIRED status in Golden Copy files are: {:,} ".format(retired_level_1.shape[0]))

The total records with RETIRED status in Golden Copy files are: 217,312 


In the Golden Copy, GLEIF allows up to 5 Legal Entity Events to be stored per entity. As we are focussing on RETIRED entities, the code below retrieves not only Legal Entity Event data, but also considers successor information. 

In case an entity ceased to exist, it may be succeeded by another entity. The successor information is given in the SuccessorLEI field, if the successor has an LEI. If the successor does not have an LEI, its entity name is given via the SuccessorEntityName field.

In [ ]:
# Sample RETIRED entities and keep 100 samples
retired_df = retired_level_1.sample(n=100, random_state=42)

# Identify Legal Entity Event (LEE) columns
lee_cols = ColumnNames.EVENT_COLUMNS

print(f"Found {len(lee_cols)} Legal Entity Event columns")

# Flatten events into a long table (one row per event field)
records = []

for _, row in retired_df.iterrows():
    lei = row["LEI"]
    reg_status = row["Registration.RegistrationStatus"]
    last_update = row.get("Registration.LastUpdateDate", pd.NA)
    successor_lei = row.get("Entity.SuccessorEntity.1.SuccessorLEI", pd.NA)
    successor_name = row.get("Entity.SuccessorEntity.1.SuccessorEntityName", pd.NA)

    for col in lee_cols:
        val = row[col]
        if pd.isna(val):
            continue

        parts = col.split(".")
        
        try:
            idx = parts.index("LegalEntityEvent")
        except ValueError:
            # Column is not in the expected pattern; skip it
            continue

        event_index = parts[idx + 1]            # e.g. "1"
        field_name = ".".join(parts[idx + 2:])  # e.g. "LegalEntityEventType", "event_status"

        records.append({
            "LEI": lei,
            "RegistrationStatus": reg_status,
            "LastUpdateDate": last_update,
            "SuccessorLEI": successor_lei,
            "SuccessorName": successor_name,
            "event_index": int(event_index),
            "field_name": field_name,
            "value": val,
        })

events_long = pd.DataFrame(records)
print("events_long shape:", events_long.shape)

# Pivot to a wide event-level table
index_cols = [
    "LEI",
    "RegistrationStatus",
    "LastUpdateDate",
    "event_index",
]

events_wide = (
    events_long
    .pivot_table(
        index=index_cols,
        columns="field_name",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

events_wide.columns.name = None
print("events_wide shape:", events_wide.shape)

# Add successor info back (one row per LEI)
successor_info = (
    events_long
    .groupby("LEI")[["SuccessorLEI", "SuccessorName"]]
    .first()
    .reset_index()
)

events_wide = events_wide.merge(successor_info, on="LEI", how="left")


Found 20 Legal Entity Event columns
events_long shape: (468, 8)
events_wide shape: (117, 8)


Using the EventType field, Legal Entity Events can be categorized. Below, we focus on EventTypes that usually lead to the retirement of an entity, and may involve a successor entity.

In [ ]:
# Event types that may result in retirement and a potential successor
trigger_terms = [
    "ABSORPTION",
    "ACQUISITION_BRANCH",
    "MERGERS_AND_ACQUISITIONS",
    "BREAKUP",
    "SPINOFF",
    "DEMERGER",
    "BANKRUPTCY",
    "INSOLVENCY",
    "LIQUIDATION",
    "DISSOLUTION"
]

pattern = "|".join(trigger_terms)

events_wide["retirement_related"] = events_wide["LegalEntityEventType"].str.contains(
    pattern,
    regex=True,
    na=False,
)

The event_status element can hold one of the following values: "COMPLETED", "IN_PROGRESS" or "WITHDRAWN_CANCELLED". For analyzing RETIRED entities, we want to focus on COMPLETED Legal Entity Events.

In [ ]:
# Consider the events that are COMPLETED
events_wide_complete = events_wide.loc[events_wide['event_status']=='COMPLETED']

retirement_events = events_wide_complete.loc[events_wide_complete["retirement_related"]].copy()
print("Retirement-related events:", retirement_events.shape)
display(retirement_events.head())

Retirement-related events: (98, 11)


,LEI,RegistrationStatus,LastUpdateDate,event_index,LegalEntityEventEffectiveDate,LegalEntityEventRecordedDate,LegalEntityEventType,event_status,SuccessorLEI,SuccessorName,retirement_related
0,2138001JNNHCO1Z37J12,RETIRED,2025-11-19T17:48:51+01:00,1,2025-09-24T00:00:00+02:00,2025-11-19T17:48:51+01:00,DISSOLUTION,COMPLETED,None,None,True
1,2138004EDVLAT7JZ8398,RETIRED,2024-12-16T17:06:10.993Z,1,2021-10-25T00:00:00Z,2022-03-25T00:00:00Z,DISSOLUTION,COMPLETED,None,None,True
2,21380066RAWJDO754570,RETIRED,2024-12-16T17:06:10.993Z,1,2024-08-19T00:00:00Z,2024-08-19T14:20:47Z,DISSOLUTION,COMPLETED,None,None,True
3,2138006HLECCVLBWCO77,RETIRED,2024-12-16T17:06:10.993Z,1,2023-10-31T00:00:00Z,2023-11-16T10:46:54Z,DISSOLUTION,COMPLETED,None,None,True
4,2138007WD4MZM5XN7P42,RETIRED,2024-12-16T17:06:10.993Z,1,2021-02-16T00:00:00Z,2023-11-17T11:57:18Z,DISSOLUTION,COMPLETED,None,None,True


In [ ]:
retirement_events[retirement_events['SuccessorLEI'].notna()]

,LEI,RegistrationStatus,LastUpdateDate,event_index,LegalEntityEventEffectiveDate,LegalEntityEventRecordedDate,LegalEntityEventType,event_status,SuccessorLEI,SuccessorName,retirement_related
10,213800QYX95B24WF7X18,RETIRED,2024-12-16T17:06:10.993Z,1,2018-12-17T00:00:00Z,2022-03-25T00:00:00Z,MERGERS_AND_ACQUISITIONS,COMPLETED,2138007PIAXK3C4EZP58,None,True
19,259400VDL4HVEETZUO24,RETIRED,2023-08-02T15:36:13+02:00,1,2023-02-17T00:00:00+01:00,2023-08-02T15:36:13+02:00,MERGERS_AND_ACQUISITIONS,COMPLETED,259400B5YFPKIOILLH25,None,True
20,3157001MB75ZIRNMY368,RETIRED,2024-06-15T08:52:20+02:00,1,2015-03-23T00:00:00+01:00,2022-03-18T00:00:00+01:00,MERGERS_AND_ACQUISITIONS,COMPLETED,315700X8JQ3IU0NGK501,None,True
22,3358003HBSTJZ8K84S12,RETIRED,2023-10-10T04:19:18.03Z,1,2023-06-21T18:30:00.001Z,2023-10-10T04:19:18.03Z,MERGERS_AND_ACQUISITIONS,COMPLETED,335800EK6HZSC4CVAJ09,None,True
26,52990004G2T8SQHFCY43,RETIRED,2023-02-03T15:45:41+01:00,1,2022-02-24T00:00:00+01:00,2023-02-03T15:45:41+01:00,MERGERS_AND_ACQUISITIONS,COMPLETED,529900K3FKR7CM1DUW63,None,True
36,54930010HPT5IFDMHI17,RETIRED,2024-12-16T17:06:10.993Z,1,2023-04-17T00:00:00Z,2023-11-06T11:21:27Z,MERGERS_AND_ACQUISITIONS,COMPLETED,549300OTZPIXC0ED6844,None,True
38,549300438JG7EP075C16,RETIRED,2023-08-04T15:46:02.115Z,1,2020-06-01T00:00:00.000Z,2022-03-24T19:41:40.000Z,MERGERS_AND_ACQUISITIONS,COMPLETED,549300FOHR7SBPQ7WJ92,None,True
45,549300IHG2IIKY7LT759,RETIRED,2023-08-16T12:52:00Z,1,2020-06-16T02:00:00+02:00,2022-03-24T20:41:40+01:00,MERGERS_AND_ACQUISITIONS,COMPLETED,5493006ID4LFX21UEW81,None,True
74,74370033X2BDDOHHYH87,RETIRED,2022-03-31T14:24:50+03:00,1,2021-03-31T00:00:00+03:00,2021-03-31T00:00:00+03:00,MERGERS_AND_ACQUISITIONS,COMPLETED,743700KKSGN4OMWDGL22,None,True
75,7437006AOVHI6NONEV08,RETIRED,2025-05-31T21:10:01+03:00,1,2024-05-31T00:00:00+03:00,2024-05-31T00:00:00+03:00,MERGERS_AND_ACQUISITIONS,COMPLETED,743700G2HZTFJZIOXR78,None,True


### Time difference between event occurrence and reporting

The code below analyzes the gap between the EffectiveDate and the RecordedDate of an event. The EffectiveDate describes when the event had an effect on the entity, the RecordedDate indicates when it was recorded in the Global LEI System.

In [ ]:

def extract_date_only(col):
    # convert to string & strip
    s = col.astype("string").str.strip()

    dates_str = s.str.extract(r"(\d{4}-\d{2}-\d{2})")[0]

    # parse to keep date only
    return pd.to_datetime(dates_str, format="%Y-%m-%d", errors="coerce")


# apply to the date columns
retirement_events["EffectiveDate_date"] = extract_date_only(
    retirement_events["LegalEntityEventEffectiveDate"]
)
retirement_events["RecordedDate_date"] = extract_date_only(
    retirement_events["LegalEntityEventRecordedDate"]
)

# difference in days
retirement_events["retirement_delay_days"] = (
    retirement_events["RecordedDate_date"] - retirement_events["EffectiveDate_date"]
).dt.days


In [ ]:

retirement_events[[
    "LEI",
    "LegalEntityEventType",
    "RecordedDate_date",
    "EffectiveDate_date",
    "retirement_delay_days",
]].sample(3)

,LEI,LegalEntityEventType,RecordedDate_date,EffectiveDate_date,retirement_delay_days
107,969500KX9E5XM1HVQX82,DISSOLUTION,2022-03-23,2022-03-01,22
3,2138006HLECCVLBWCO77,DISSOLUTION,2023-11-16,2023-10-31,16
105,969500F5DK5T7KWPD873,DISSOLUTION,2023-01-05,2023-01-05,0


#### <p style="text-align: left;">Resources:</p>

GLEIF Explains video on Legal Entity Events: https://www.gleif.org/en/newsroom/gleif-podcasts/gleif-explains-new-lei-data-formats-legal-entity-events-in-the-global-lei-system

Version 1.0.0 <br> 
Last updated: 5 December 2025